# IPL Match Analysis using SQL

## Problem Statement
Analyze IPL match data using SQL queries to extract insights about team performance, toss impact, venue statistics, and player achievements.

## Approach
1. Load IPL match data into SQLite database
2. Run analytical SQL queries
3. Visualize results

## Step 1: Import Libraries and Setup Database

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Step 2: Load Data into SQLite

In [ ]:
# Read CSV file
df = pd.read_csv('matches.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
# Create SQLite database and load data
conn = sqlite3.connect('ipl_data.db')

# Load dataframe into SQL table
df.to_sql('matches', conn, if_exists='replace', index=False)

# Verify data loaded
result = conn.execute("SELECT COUNT(*) FROM matches").fetchone()
print(f"Database created successfully! Total records: {result[0]}")

# Show table schema
print("\nTable Schema:")
schema = conn.execute("PRAGMA table_info(matches)").fetchall()
for col in schema:
    print(f"  {col[1]:20s} {col[2]}")

## Step 3: SQL Queries for Analysis

We will run 10 SQL queries to analyze different aspects of IPL matches.

### Query 1: Total Matches Per Season

In [ ]:
query1 = """
SELECT season, COUNT(*) as total_matches
FROM matches
GROUP BY season
ORDER BY season
"""

df_season = pd.read_sql_query(query1, conn)
print("Matches Per Season:")
print(df_season.to_string(index=False))

# Visualization
plt.figure(figsize=(12, 5))
plt.bar(df_season['season'], df_season['total_matches'], color='steelblue')
plt.xlabel('Season')
plt.ylabel('Number of Matches')
plt.title('IPL Matches Per Season')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Query 2: Top Winning Teams Overall

In [ ]:
query2 = """
SELECT winner, COUNT(*) as total_wins
FROM matches
WHERE winner IS NOT NULL
GROUP BY winner
ORDER BY total_wins DESC
"""

df_teams = pd.read_sql_query(query2, conn)
print("Top Winning Teams:")
print(df_teams.to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
colors = sns.color_palette('viridis', len(df_teams))
plt.barh(df_teams['winner'], df_teams['total_wins'], color=colors)
plt.xlabel('Total Wins')
plt.ylabel('Team')
plt.title('IPL Teams by Total Wins')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Query 3: Toss Decision Impact on Winning

In [ ]:
query3 = """
SELECT 
    toss_decision,
    COUNT(*) as total_tosses,
    SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) as toss_winner_won_match,
    ROUND(SUM(CASE WHEN toss_winner = winner THEN 1.0 ELSE 0 END) * 100.0 / COUNT(*), 2) as win_percentage
FROM matches
WHERE toss_winner IS NOT NULL AND winner IS NOT NULL
GROUP BY toss_decision
"""

df_toss = pd.read_sql_query(query3, conn)
print("Toss Decision Impact:")
print(df_toss.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart for toss decisions
axes[0].pie(df_toss['total_tosses'], labels=df_toss['toss_decision'], 
            autopct='%1.1f%%', colors=['#ff9999','#66b3ff'], startangle=90)
axes[0].set_title('Toss Decision Distribution')

# Bar chart for win percentage
axes[1].bar(df_toss['toss_decision'], df_toss['win_percentage'], color=['#ff9999','#66b3ff'])
axes[1].set_xlabel('Toss Decision')
axes[1].set_ylabel('Win Percentage (%)')
axes[1].set_title('Win % After Toss Decision')
for i, v in enumerate(df_toss['win_percentage']):
    axes[1].text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### Query 4: Match Results Distribution (Wins by Runs vs Wickets)

In [ ]:
query4 = """
SELECT result, COUNT(*) as count
FROM matches
WHERE result IS NOT NULL
GROUP BY result
ORDER BY count DESC
"""

df_result = pd.read_sql_query(query4, conn)
print("Match Results Distribution:")
print(df_result.to_string(index=False))

# Visualization
plt.figure(figsize=(8, 5))
plt.pie(df_result['count'], labels=df_result['result'], 
        autopct='%1.1f%%', colors=sns.color_palette('Set2'), startangle=90)
plt.title('Match Results Distribution')
plt.tight_layout()
plt.show()

### Query 5: Top 10 Venues by Number of Matches

In [ ]:
query5 = """
SELECT venue, COUNT(*) as total_matches
FROM matches
GROUP BY venue
ORDER BY total_matches DESC
LIMIT 10
"""

df_venue = pd.read_sql_query(query5, conn)
print("Top 10 Venues:")
print(df_venue.to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
plt.barh(df_venue['venue'], df_venue['total_matches'], color='coral')
plt.xlabel('Number of Matches')
plt.ylabel('Venue')
plt.title('Top 10 IPL Venues')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Query 6: Player of the Match Leaders

In [ ]:
query6 = """
SELECT player_of_match, COUNT(*) as awards
FROM matches
WHERE player_of_match IS NOT NULL
GROUP BY player_of_match
ORDER BY awards DESC
LIMIT 10
"""

df_potm = pd.read_sql_query(query6, conn)
print("Player of the Match Leaders:")
print(df_potm.to_string(index=False))

# Visualization
plt.figure(figsize=(10, 6))
plt.bar(df_potm['player_of_match'], df_potm['awards'], color='gold', edgecolor='black')
plt.xlabel('Player')
plt.ylabel('Number of Awards')
plt.title('Top 10 Player of the Match Awards')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Query 7: Head-to-Head Team Matchups

In [ ]:
query7 = """
SELECT 
    team1,
    team2,
    COUNT(*) as matches_played,
    SUM(CASE WHEN winner = team1 THEN 1 ELSE 0 END) as team1_wins,
    SUM(CASE WHEN winner = team2 THEN 1 ELSE 0 END) as team2_wins
FROM matches
WHERE winner IS NOT NULL
GROUP BY team1, team2
HAVING matches_played >= 5
ORDER BY matches_played DESC
LIMIT 15
"""

df_h2h = pd.read_sql_query(query7, conn)
print("Head-to-Head Matchups (Top 15):")
print(df_h2h.to_string(index=False))

### Query 8: Season-wise Top Performing Team

In [ ]:
query8 = """
SELECT season, winner, wins
FROM (
    SELECT 
        season,
        winner,
        COUNT(*) as wins,
        ROW_NUMBER() OVER (PARTITION BY season ORDER BY COUNT(*) DESC) as rank
    FROM matches
    WHERE winner IS NOT NULL
    GROUP BY season, winner
) subq
WHERE rank = 1
ORDER BY season
"""

df_season_top = pd.read_sql_query(query8, conn)
print("Season-wise Top Team:")
print(df_season_top.to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
plt.bar(df_season_top['season'].astype(str), df_season_top['wins'], color='purple')
plt.xlabel('Season')
plt.ylabel('Wins')
plt.title('Season-wise Top Team Wins')
plt.xticks(rotation=45)

# Add team names on bars
for i, (season, team, wins) in enumerate(zip(df_season_top['season'], df_season_top['winner'], df_season_top['wins'])):
    plt.text(i, wins + 0.2, team, ha='center', fontsize=8, rotation=45)

plt.tight_layout()
plt.show()

### Query 9: City-wise Match Statistics

In [ ]:
query9 = """
SELECT 
    city,
    COUNT(*) as total_matches,
    ROUND(AVG(target_runs), 2) as avg_target,
    ROUND(AVG(result_margin), 2) as avg_margin
FROM matches
WHERE city IS NOT NULL
GROUP BY city
HAVING total_matches >= 5
ORDER BY total_matches DESC
LIMIT 10
"""

df_city = pd.read_sql_query(query9, conn)
print("City-wise Statistics:")
print(df_city.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Total matches by city
axes[0].barh(df_city['city'], df_city['total_matches'], color='teal')
axes[0].set_xlabel('Total Matches')
axes[0].set_title('Matches by City')
axes[0].invert_yaxis()

# Average target by city
axes[1].barh(df_city['city'], df_city['avg_target'], color='orange')
axes[1].set_xlabel('Average Target Runs')
axes[1].set_title('Average Target by City')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Query 10: Teams Winning Toss and Match Together

In [ ]:
query10 = """
SELECT 
    toss_winner,
    toss_decision,
    COUNT(*) as count
FROM matches
WHERE toss_winner = winner
GROUP BY toss_winner, toss_decision
ORDER BY count DESC
"""

df_toss_win = pd.read_sql_query(query10, conn)
print("Teams Winning Toss and Match:")
print(df_toss_win.to_string(index=False))

# Visualization
plt.figure(figsize=(12, 6))
pivot = df_toss_win.pivot(index='toss_winner', columns='toss_decision', values='count').fillna(0)
pivot.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Set2')
plt.xlabel('Team')
plt.ylabel('Count')
plt.title('Teams Winning Toss + Match (by Decision)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Toss Decision')
plt.tight_layout()
plt.show()

## Step 4: Summary of Findings

In [ ]:
# Close the database connection
conn.close()

print("=" * 60)
print("IPL SQL ANALYSIS COMPLETE")
print("=" * 60)
print(f"Database: ipl_data.db")
print(f"Total Queries Executed: 10")
print("\nKey Insights:")
print("1. Matches per season trend analyzed")
print("2. Top winning teams identified")
print("3. Toss decision impact measured")
print("4. Match result distribution mapped")
print("5. Top venues ranked")
print("6. Player of match leaders found")
print("7. Head-to-head matchups computed")
print("8. Season-wise best teams found")
print("9. City-wise statistics computed")
print("10. Toss + match win patterns discovered")